# Phase 10: Prediction Application

**Framework Used:** Streamlit

**Features Implemented:**
- Attractive Modern UI
- Sidebar Navigation
- Multiple Pages (Home, Overview, EDA, Model Performance, Chatbot)
- Real-time Chat Interface
- Error Handling
- Model Loading using Joblib
- Clean & Professional Design

**Screenshots:** Saved in `screenshots/` folder

In [ ]:
import streamlit as st
import joblib
import pandas as pd
import os
from datetime import datetime

# ================== YOUR PATHS ==================
MODELS_PATH = r'C:\Users\hemanjali\Desktop\clg enquiry chatbot\models'
# ===============================================

st.set_page_config(
    page_title="College Enquiry AI Chatbot",
    page_icon="🎓",
    layout="wide"
)

# Custom CSS for Attractive UI
st.markdown("""
<style>
    .main-header {font-size: 40px; color: #1E3A8A; text-align: center;}
    .stButton>button {width: 100%; background-color: #1E3A8A; color: white;}
    .success-msg {color: green; font-weight: bold;}
</style>
""", unsafe_allow_html=True)

# Load Model and Vectorizer
@st.cache_resource
def load_model():
    model = joblib.load(f'{MODELS_PATH}/chatbot_model.pkl')
    vectorizer = joblib.load(f'{MODELS_PATH}/tfidf_vectorizer.pkl')
    encoder = joblib.load(f'{MODELS_PATH}/label_encoder.pkl')
    return model, vectorizer, encoder

model, vectorizer, encoder = load_model()

# Load intents for responses
import json
with open('data/intents.json', 'r', encoding='utf-8') as f:
    intents_data = json.load(f)

response_dict = {item['tag']: item['responses'] for item in intents_data['intents']}

# Sidebar Navigation
st.sidebar.title("🎓 Navigation")
page = st.sidebar.radio("Go to", 
    ["🏠 Home", "📊 Project Overview", "📈 EDA Visualizations", 
     "📊 Model Performance", "💬 Chatbot"])

# ====================== PAGES ======================

if page == "🏠 Home":
    st.title("🎓 College Enquiry AI Chatbot")
    st.markdown("### Welcome to Our Intelligent College Assistant!")
    st.image("https://source.unsplash.com/random/800x400/?college", use_column_width=True)
    
    col1, col2, col3 = st.columns(3)
    with col1:
        st.metric("Accuracy", "> 95%")
    with col2:
        st.metric("Intents", "38+")
    with col3:
        st.metric("Response Time", "< 1 sec")

elif page == "📊 Project Overview":
    st.title("Project Overview")
    st.write("""
    This AI Chatbot helps students and parents get instant answers about:
    - Admissions
    - Fees Structure
    - Courses
    - Placements
    - Hostel & Facilities
    - Scholarships and more...
    """)
    st.success("Built with Streamlit + Machine Learning")

elif page == "📈 EDA Visualizations":
    st.title("Exploratory Data Analysis")
    st.image(f"{r'C:\\Users\\hemanjali\\Desktop\\clg enquiry chatbot\\screenshots'}\\eda\\intent_distribution.png", 
             caption="Intent Distribution", use_column_width=True)
    st.image(f"{r'C:\\Users\\hemanjali\\Desktop\\clg enquiry chatbot\\screenshots'}\\eda\\wordcloud.png", 
             caption="Word Cloud", use_column_width=True)

elif page == "📊 Model Performance":
    st.title("Model Performance")
    st.success("Final Model Accuracy: > 95%")
    st.info("Model Used: Optimized Logistic Regression")
    
    # You can add more metrics here

elif page == "💬 Chatbot":
    st.title("💬 College Enquiry Chatbot")
    st.markdown("Ask me anything about the college!")

    # Initialize chat history
    if "messages" not in st.session_state:
        st.session_state.messages = []

    # Display chat history
    for message in st.session_state.messages:
        with st.chat_message(message["role"]):
            st.markdown(message["content"])

    # User Input
    if prompt := st.chat_input("Type your question here..."):
        # Add user message
        st.session_state.messages.append({"role": "user", "content": prompt})
        with st.chat_message("user"):
            st.markdown(prompt)

        # Preprocess and Predict
        try:
            # Vectorize input
            vector = vectorizer.transform([prompt])
            prediction = model.predict(vector)[0]
            intent = encoder.inverse_transform([prediction])[0]

            # Get response
            responses = response_dict.get(intent, ["Sorry, I couldn't understand your question."])
            reply = responses[0]

            # Add bot response
            st.session_state.messages.append({"role": "assistant", "content": reply})
            with st.chat_message("assistant"):
                st.markdown(reply)

        except Exception as e:
            with st.chat_message("assistant"):
                st.error("Sorry, something went wrong. Please try again.")

# Footer
st.sidebar.markdown("---")
st.sidebar.info("Made with ❤️ for College Enquiry System")
st.sidebar.caption("B.Tech Final Project")